In [56]:
import psi4
import pandas as pd
import os
import numpy as np
import os
import sys
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)
%load_ext autoreload
%autoreload 2
from src.lps_rscf import lps_solver
from src.sic_tf import *
from src.sic_lda import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
csv_file = '../data/sic_closed_shell_atoms.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 48 rows found.


In [72]:
ATOMS = {
    'He':  {'mult': 1,'N': 2}, 
    'Be': {'mult': 1,'N': 4},
    'Ne': {'mult': 1,'N': 10}, 
    'Mg': {'mult': 1,'N': 12},
    'Ar':  {'mult': 1,'N': 18}, 
    'Ca':  {'mult': 1,'N': 20},
    'Zn':  {'mult': 1,'N': 30}, 
    'Kr':  {'mult': 1,'N': 36}
}
psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

METHOD = 'TF_GR W LDA'
MAX_ITER = 2000
DAMPING = [0.9, 0.0, 0.001]
DIIS_OPT = [True, True]

for atom in ATOMS:
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    mol = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations, e_list, e_conv, d_conv = lps_solver(
            mol=mol,
            E_conv=1.0e-5,
            D_conv=1.0e-5,
            maxiter=MAX_ITER,
            TP=['LDA_K_TF', sic_gr(ATOMS[atom]['N'])],
            lam=1.0,
            EXC=['LDA_X', 1.0, 'LDA_C_VWN', 0.0],
            FA=[False, 1.0],
            damp=DAMPING,
            DIIS=DIIS_OPT,
            Guess=None,
            lehtomaki=False,
            verbose=False
        )
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print('\nFinal SCF energy: %.4f Hartree' \
                '\nConverged in %.i iterations' % ( E, iterations))
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "ChemPot,Ha": round(mu, 6),
                "Iterations": iterations,
                "DIIS": DIIS_OPT,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating He with TF_GR W LDA...

Final SCF energy: -2.7236 Hartree
Converged in 39 iterations
Calculating Be with TF_GR W LDA...

Final SCF energy: -15.0608 Hartree
Converged in 69 iterations
Calculating Ne with TF_GR W LDA...

Final SCF energy: -134.2432 Hartree
Converged in 102 iterations
Calculating Mg with TF_GR W LDA...

Final SCF energy: -206.9697 Hartree
Converged in 114 iterations
Calculating Ar with TF_GR W LDA...

Final SCF energy: -542.0543 Hartree
Converged in 148 iterations
Calculating Ca with TF_GR W LDA...

Final SCF energy: -696.1797 Hartree
Converged in 154 iterations
Calculating Zn with TF_GR W LDA...

Final SCF energy: -1823.9810 Hartree
Converged in 210 iterations
Calculating Kr with TF_GR W LDA...

Final SCF energy: -2812.5981 Hartree
Converged in 253 iterations


In [73]:
df.to_csv(csv_file, index=False)
# df = df[~df['Method'].str.contains('TF_GR W FA D_B', na=False)]

In [7]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [75]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['TF_B W FA', 'TF_GR W LDA', 'TF_GR W FA', 'TF_ABSP2 W FA', 'TF_ABSP1 W FA', 'TFW FA', 'W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)
display(energy_table)

,TF_B W FA,TF_GR W LDA,TF_GR W FA,TF_ABSP2 W FA,TF_ABSP1 W FA,TFW FA,W FA,RHF/UGBS
He,-2.861680,-2.723640,-2.861680,-3.016522,-3.210491,-1.539225,-2.861680,-2.861680
Be,-12.706809,-15.060804,-14.790827,-15.590172,-16.513037,-8.318528,-19.019442,-14.573023
Ne,-106.209206,-134.243152,-129.491096,-139.590305,-145.952630,-82.790109,-264.351611,-128.547083
Mg,-163.896738,-206.969712,-199.806807,-215.922834,-225.154209,-131.025072,-450.782450,-199.614621
Ar,-433.010279,-542.054274,-525.287421,-569.078003,-590.061891,-363.016581,-1487.979296,-526.817486
Ca,-557.937756,-696.179671,-675.476822,-731.823759,-757.773231,-472.757119,-2032.047374,-676.758154
Zn,-1483.084038,-1823.980983,-1778.600588,-1923.778678,-1982.409444,-1302.039741,-6766.870137,-1777.848060
Kr,-2303.202334,-2812.598123,-2748.699338,-2968.555726,-3053.068173,-2049.433009,-11640.920803,-2752.054860
MAE(Ha),127.020000,19.380000,1.030000,61.040000,86.880000,208.520000,2073.220000,NaN
rMAE(%),14.540000,3.360000,0.370000,7.670000,12.280000,34.080000,156.040000,NaN


In [76]:
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
mu_table = mu_table[new_order]

reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

display(mu_table)

,TF_B W FA,TF_GR W LDA,TF_GR W FA,TF_ABSP2 W FA,TF_ABSP1 W FA,TFW FA,W FA,RHF/UGBS
He,-0.917956,-0.516968,-0.917956,-1.019353,-1.154642,-0.285218,-0.917956,-0.917956
Be,-0.788604,-0.740194,-1.118392,-1.264758,-1.447818,-0.319222,-2.022702,-0.309271
Ne,-0.586168,-0.688651,-0.970804,-1.189634,-1.345798,-0.333823,-7.541360,-0.850411
Mg,-0.550891,-0.649220,-0.910327,-1.124347,-1.263974,-0.333141,-10.119237,-0.253048
Ar,-0.484604,-0.561940,-0.778713,-0.964294,-1.065751,-0.329265,-20.070485,-0.590989
Ca,-0.470124,-0.541495,-0.747621,-0.923780,-1.016159,-0.327909,-24.126936,-0.195527
Zn,-0.423649,-0.475265,-0.644615,-0.784569,-0.848065,-0.322032,-49.955267,-0.292463
Kr,-0.406844,-0.451680,-0.606726,-0.732036,-0.785730,-0.319229,-69.889360,-0.524161
MAE(Ha),0.210000,0.250000,0.350000,0.510000,0.620000,0.230000,22.590000,NaN
rMAE(%),66.180000,77.100000,123.230000,168.470000,198.950000,40.720000,6373.720000,NaN
